# Experiment 5: Pause inside the look-around part, change one head, resume

The look-around part (attention) of each block is made of **12 heads** working side by side.
Each head, for every word:
1. decides **where to look** among the earlier words (its attention weights, adding up to 100%),
2. collects a summary of what it saw: **64 numbers** (its head output).

The 12 head outputs (12 x 64 = 768 numbers) are then **merged** into one look-around result and added to the table.
In Experiment 2 we changed the whole merged result. Here we go one level deeper and change **one head at a time**.

In this notebook we:
1. Rebuild the look-around part head by head, and check it gives exactly the normal answer.
2. **PAUSE** inside a block, after the heads have run but before they are merged, and **save** the 12 head outputs.
3. Show where each head of ' speak' looks, and which heads push towards ' French'.
4. **MODIFY** one head's output for ' speak': switch it off, or replace it with the head output from the Berlin sentence.
5. **RESUME** by merging the heads and running the rest of the model, **without rerunning** anything earlier.
6. Map all 144 heads (12 blocks x 12 heads), and find the few that carry "Paris -> French".
7. Follow up on Experiment 4: why did swapping only the block 11 **keys** make ' French' *more* likely?

- **Part A:** block 8, every head, row of ' speak'
- **Part B:** all 144 heads, and the top heads changed together
- **Part C:** block 11, changing where the heads look

## Step 0: Install (Colab only)
Tested with PyTorch 2.14.1 and Transformers 5.20.2. Uncomment only if something breaks.

In [ ]:
# !pip install -q transformers==5.20.2

## Step 1: Load GPT-2 small

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_grad_enabled(False)  # prediction only, no training
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
gpt2 = model.transformer
NUM_BLOCKS = len(gpt2.h)
NUM_HEADS = model.config.n_head
WIDTH = model.config.n_embd
HEAD_SIZE = WIDTH // NUM_HEADS

print("Device:", device)
print("Number of blocks:", NUM_BLOCKS)
print(f"Heads per block: {NUM_HEADS}, numbers per head: {HEAD_SIZE} ({NUM_HEADS} x {HEAD_SIZE} = {WIDTH})")

Device: cpu
Number of blocks: 12
Heads per block: 12, numbers per head: 64 (12 x 64 = 768)


## Step 2: Normal run (no changes)

In [2]:
PROMPT = "I live in Paris, so I speak"
OTHER_PROMPT = "I live in Berlin, so I speak"
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)
other_ids = tokenizer(OTHER_PROMPT, return_tensors="pt").input_ids.to(device)
assert other_ids.shape == input_ids.shape
words = [tokenizer.decode(t) for t in input_ids[0]]
print("Word-pieces:", words)

PARIS_POS, LAST_POS = 3, len(words) - 1
FRENCH, GERMAN = tokenizer.encode(" French")[0], tokenizer.encode(" German")[0]

def top_predictions(logits, k=5):
    probs = logits[0, -1].softmax(-1)
    values, indices = probs.topk(k)
    return [(tokenizer.decode(i), v.item()) for i, v in zip(indices, values)]

def chances(logits):
    p = logits[0, -1].softmax(-1)
    return tokenizer.decode(p.argmax()), p[FRENCH].item(), p[GERMAN].item()

def compare(new_logits, title="Modified model"):
    """Print the normal and new top 5 side by side, plus the chance of ' French' and ' German'."""
    print(f"{'Normal model':<26}{title}")
    for (w1, p1), (w2, p2) in zip(top_predictions(normal_logits), top_predictions(new_logits)):
        print(f"  {w1!r:12} {p1:6.2%}      {w2!r:12} {p2:6.2%}")
    _, f0, g0 = chances(normal_logits)
    _, f1, g1 = chances(new_logits)
    print(f"\n  ' French': {f0:6.2%} -> {f1:6.2%}")
    print(f"  ' German': {g0:6.2%} -> {g1:6.2%}")

normal_logits = model(input_ids, use_cache=False).logits
NORMAL_FRENCH = chances(normal_logits)[1]
print("\nNormal model - top 5 next words:")
for word, p in top_predictions(normal_logits):
    print(f"  {word!r:12} {p:6.2%}")

Word-pieces: ['I', ' live', ' in', ' Paris', ',', ' so', ' I', ' speak']

Normal model - top 5 next words:
  ' French'    63.05%
  ' English'    4.93%
  ' to'         3.16%
  ' a'          2.94%
  ' with'       1.76%


## Step 3: Run the look-around part head by head
We rebuild the look-around part ourselves, so we can stop between the heads and the merge:
- `qkv(i, state)` gives, for every head and word, a **query** (what this word is looking for), a **key** (what it offers)
  and a **value** (what it hands over). Each is 64 numbers.
- `attend(q, k, v)` compares queries with keys to get the **attention weights** (each word may only look at itself and earlier words),
  then mixes the values with those weights to get the **head outputs**.
- `merge_part(i, state, head_out)` joins the 12 head outputs, turns them into the 768-number look-around result, and adds it to the table.
- `think_part(i, state)` runs the think part (MLP) and adds its result.
- `run_until_heads(ids, layer)` runs blocks before `layer`, then the heads of block `layer`, then **PAUSES**.
  The **checkpoint** holds the table that entered the block, the attention weights and the 12 head outputs.
- `resume_after_heads(checkpoint)` **RESUMES**: merges the (possibly changed) heads, runs the think part, then the remaining blocks.

In [3]:
def embed(input_ids):
    positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
    return gpt2.wte(input_ids) + gpt2.wpe(positions)

def split_heads(t):
    """(1, words, 768) -> (1, 12 heads, words, 64)"""
    return t.view(t.shape[0], t.shape[1], NUM_HEADS, HEAD_SIZE).transpose(1, 2)

def qkv(i, state):
    block = gpt2.h[i]
    q, k, v = block.attn.c_attn(block.ln_1(state)).split(WIDTH, dim=2)
    return split_heads(q), split_heads(k), split_heads(v)

def attend(q, k, v):
    """Attention weights (1, heads, words, words) and head outputs (1, heads, words, 64)."""
    n = q.shape[2]
    scores = q @ k.transpose(-1, -2) / HEAD_SIZE ** 0.5
    later_words = torch.triu(torch.ones(n, n, dtype=torch.bool, device=q.device), diagonal=1)
    weights = scores.masked_fill(later_words, float("-inf")).softmax(-1)
    return weights, weights @ v

def merge_part(i, state, head_out):
    """Join the 12 heads, turn them into the look-around result, and add it to the table."""
    merged = head_out.transpose(1, 2).reshape(state.shape)
    return state + gpt2.h[i].attn.c_proj(merged)

def think_part(i, state):
    block = gpt2.h[i]
    return state + block.mlp(block.ln_2(state))

def final_logits(state):
    return model.lm_head(gpt2.ln_f(state))

def run_model(input_ids, edits=None):
    """Full run, head by head. `edits` = {block number: function that changes its head outputs}."""
    edits = edits or {}
    state = embed(input_ids)
    for i in range(NUM_BLOCKS):
        _, head_out = attend(*qkv(i, state))
        if i in edits:
            head_out = edits[i](head_out)
        state = think_part(i, merge_part(i, state, head_out))
    return final_logits(state)

def run_until_heads(input_ids, layer):
    """Run blocks 0..layer-1 fully, then the heads of block `layer`, then PAUSE (before the merge)."""
    state = embed(input_ids)
    for i in range(layer):
        _, head_out = attend(*qkv(i, state))
        state = think_part(i, merge_part(i, state, head_out))
    weights, head_out = attend(*qkv(layer, state))
    return {"layer": layer, "before_look": state, "weights": weights, "head_out": head_out}

def resume_after_heads(checkpoint):
    """RESUME: merge the (possibly changed) heads, run the think part, then all later blocks."""
    layer = checkpoint["layer"]
    state = think_part(layer, merge_part(layer, checkpoint["before_look"], checkpoint["head_out"]))
    for i in range(layer + 1, NUM_BLOCKS):
        _, head_out = attend(*qkv(i, state))
        state = think_part(i, merge_part(i, state, head_out))
    return final_logits(state)

def copy_ckpt(checkpoint, **changes):
    """Separate copy of a checkpoint, optionally with some parts replaced."""
    new = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in checkpoint.items()}
    new.update(changes)
    return new

def set_head(head_out, head, pos, new_row):
    """Copy of `head_out` with one head's output for the word at `pos` replaced."""
    head_out = head_out.clone()
    head_out[0, head, pos] = new_row
    return head_out

## Step 4: A counter that records which parts actually run
`heads 5` means the heads of block 5 ran, `merge 5` means they were merged, `think 5` means the think part of block 5 ran.

In [4]:
for h in globals().get("counter_hooks", []):  # safe to rerun this cell
    h.remove()

parts_run = []
counter_hooks = []
for i, block in enumerate(gpt2.h):
    counter_hooks.append(block.attn.c_attn.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"heads {i}")))
    counter_hooks.append(block.attn.c_proj.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"merge {i}")))
    counter_hooks.append(block.mlp.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"think {i}")))
print("Counter attached to", len(counter_hooks), "parts (12 blocks x 3 parts)")

def expected_after(layer):
    return [f"merge {layer}", f"think {layer}"] + [
        f"{p} {i}" for i in range(layer + 1, NUM_BLOCKS) for p in ("heads", "merge", "think")]

Counter attached to 36 parts (12 blocks x 3 parts)


## Step 5: Check that the head-by-head run matches the normal model

In [5]:
parts_run.clear()
split_logits = run_model(input_ids)

diff = (split_logits - normal_logits).abs().max().item()
print("Parts run:", len(parts_run), "->", parts_run[:3], "...", parts_run[-3:])
print("Largest difference from normal model:", diff)
assert diff < 1e-3, "Head-by-head run does not match the normal model"
print("OK: head-by-head run matches the normal model")

Parts run: 36 -> ['heads 0', 'merge 0', 'think 0'] ... ['heads 11', 'merge 11', 'think 11']
Largest difference from normal model: 0.0001068115234375
OK: head-by-head run matches the normal model


## Step 6: How much each head pushes towards ' French'
The merge step gives every head its own slice of numbers, so the look-around result is simply the **sum of 12 pieces, one per head** (plus a fixed offset).
`head_piece(i, h, row)` gives head `h`'s piece. We check the pieces really add up to the merged result.

As a rough guide (like in Experiment 3), we pass a piece straight through the final "numbers to words" step
and measure how much it favours ' French' over ' German'. Positive = pushes towards French. This skips the final cleanup step, so it is only an approximation.

In [6]:
FRENCH_MINUS_GERMAN = model.lm_head.weight[FRENCH] - model.lm_head.weight[GERMAN]

def head_piece(i, h, row):
    """Head h's share (768 numbers) of the look-around result of block i, for one word's 64-number head output."""
    weight = gpt2.h[i].attn.c_proj.weight  # (768 in, 768 out); head h uses input rows h*64 .. h*64+63
    return row @ weight[h * HEAD_SIZE:(h + 1) * HEAD_SIZE]

def french_push(i, h, row):
    return (head_piece(i, h, row) @ FRENCH_MINUS_GERMAN).item()

# check: the 12 pieces + the fixed offset add up to the merged look-around result
ckpt = run_until_heads(input_ids, 8)
merged = ckpt["head_out"].transpose(1, 2).reshape(1, len(words), WIDTH)
rebuilt = sum(head_piece(8, h, ckpt["head_out"][0, h, LAST_POS]) for h in range(NUM_HEADS)) + gpt2.h[8].attn.c_proj.bias
diff = (rebuilt - gpt2.h[8].attn.c_proj(merged)[0, LAST_POS]).abs().max().item()
print("Largest difference between the sum of the 12 pieces and the merged result:", diff)
assert diff < 1e-3
print("OK: the look-around result is exactly the sum of one piece per head")

Largest difference between the sum of the 12 pieces and the merged result: 1.430511474609375e-06
OK: the look-around result is exactly the sum of one piece per head


---
# Part A: block 8, every head, row of ' speak'
In Experiment 2, block 8 was the first block where ' speak' looks strongly at ' Paris', and in Experiment 4 it was one of the two blocks whose memory mattered.

## Step 7: PAUSE after the heads of block 8, before they are merged

In [7]:
PAUSE_A = 8

parts_run.clear()
ckpt_a = run_until_heads(input_ids, PAUSE_A)  # <-- PAUSE happens here
saved_a = copy_ckpt(ckpt_a)                   # <-- SAVE a separate copy
other_a = run_until_heads(other_ids, PAUSE_A)  # same pause point for the Berlin sentence

print(f"PAUSED inside block {PAUSE_A}, after its heads (not merged yet)")
print("Parts run so far:", parts_run[-2:], f"... ({len(parts_run)} parts in total)")
print("Head outputs size:", tuple(saved_a["head_out"].shape), " (1 sentence, 12 heads, 8 word-pieces, 64 numbers each)")

PAUSED inside block 8, after its heads (not merged yet)
Parts run so far: ['think 7', 'heads 8'] ... (50 parts in total)
Head outputs size: (1, 12, 8, 64)  (1 sentence, 12 heads, 8 word-pieces, 64 numbers each)


## Step 8: Where does each head of ' speak' look, and what does it push for?
For the word ' speak' in block 8: the share of attention each head gives to ' Paris', the word it looks at most,
and its push towards ' French' (Paris sentence vs Berlin sentence).

In [8]:
def describe_heads(ckpt, other):
    i = ckpt["layer"]
    print(f"{'Head':<6}{'looks at Paris':>15}{'looks most at':>16}{'French push':>13}{'(Berlin)':>10}")
    for h in range(NUM_HEADS):
        w = ckpt["weights"][0, h, LAST_POS]
        most = words[w.argmax().item()]
        push = french_push(i, h, ckpt["head_out"][0, h, LAST_POS])
        push_berlin = french_push(i, h, other["head_out"][0, h, LAST_POS])
        print(f"{h:<6}{w[PARIS_POS].item():>15.0%}{most!r:>16}{push:>13.2f}{push_berlin:>10.2f}")

describe_heads(saved_a, other_a)

Head   looks at Paris   looks most at  French push  (Berlin)
0                  3%             'I'         0.38      0.40
1                  1%             'I'        -0.07     -0.10
2                 10%             'I'         0.34      0.24
3                 16%             'I'        -0.28     -0.21
4                  1%             'I'        -0.14     -0.16
5                  2%             'I'         0.16      0.19
6                  2%             'I'        -0.00     -0.02
7                  1%             'I'        -0.07     -0.08
8                 18%             'I'         0.44      0.35
9                 10%             'I'        -0.10     -0.16
10                13%             'I'        -0.05     -0.13
11                49%        ' Paris'        11.18    -12.42


## Step 9: RESUME with the saved heads unchanged (check)
Only the merge and think part of block 8, and blocks 9 to 11, should run.

In [9]:
parts_run.clear()
resumed = resume_after_heads(copy_ckpt(saved_a))  # <-- RESUME

print("RESUMED. Parts run:", parts_run[:3], "...", parts_run[-3:], f"({len(parts_run)} parts)")
print("Largest difference from normal model:", (resumed - normal_logits).abs().max().item())
assert parts_run == expected_after(PAUSE_A), "Something earlier was rerun!"
assert torch.allclose(resumed, normal_logits, atol=1e-3)
print("OK: resuming gives the normal answer, and the heads of block 8 were not rerun")

RESUMED. Parts run: ['merge 8', 'think 8', 'heads 9'] ... ['heads 11', 'merge 11', 'think 11'] (11 parts)
Largest difference from normal model: 0.0001068115234375
OK: resuming gives the normal answer, and the heads of block 8 were not rerun


## Step 10: MODIFY — change one head of ' speak' at a time
For each head of block 8 we try two changes to the ' speak' row, each as a separate resume from the same saved checkpoint:
- **switched off:** that head's 64 numbers set to zero,
- **Berlin version:** that head's 64 numbers replaced with the same head's output in the Berlin sentence.

In [10]:
def resume_with_head(saved, head, new_row, pos=LAST_POS):
    parts_run.clear()
    logits = resume_after_heads(copy_ckpt(saved, head_out=set_head(saved["head_out"], head, pos, new_row)))
    assert parts_run == expected_after(saved["layer"])
    return logits

print(f"{'Head':<6}{'Switched off':>16}{'Berlin version':>18}")
print(f"{'':<6}{'French/German':>16}{'French/German':>18}")
for h in range(NUM_HEADS):
    _, f_off, g_off = chances(resume_with_head(saved_a, h, 0))
    _, f_swap, g_swap = chances(resume_with_head(saved_a, h, other_a["head_out"][0, h, LAST_POS]))
    print(f"{h:<6}{f_off:>10.1%}/{g_off:<5.1%}{f_swap:>12.1%}/{g_swap:<5.1%}")
_, f, g = chances(normal_logits)
print(f"\nNormal model: French {f:.1%} / German {g:.1%}")

Head      Switched off    Berlin version
         French/German     French/German
0          62.7%/0.6%        63.1%/0.6% 
1          62.7%/0.6%        63.1%/0.6% 
2          63.2%/0.6%        63.5%/0.7% 
3          66.3%/0.6%        62.9%/0.6% 
4          64.8%/0.6%        63.0%/0.6% 
5          61.1%/0.7%        62.9%/0.6% 
6          63.3%/0.6%        62.9%/0.6% 
7          62.8%/0.6%        63.1%/0.6% 
8          67.9%/0.7%        64.4%/0.6% 
9          59.9%/0.6%        63.0%/0.6% 
10         55.7%/0.6%        61.9%/0.6% 
11         50.6%/0.6%        53.4%/3.0% 

Normal model: French 63.0% / German 0.6%


---
# Part B: all 144 heads

## Step 11: Swap in the Berlin version of one head at a time, in every block
Each number is the chance of ' French' (in %) after replacing **one** head's output for ' speak' with its Berlin version.
The normal model gives 63%. Lower = that head was carrying the Paris information.

In [11]:
french_after_swap = torch.zeros(NUM_BLOCKS, NUM_HEADS)
german_after_swap = torch.zeros(NUM_BLOCKS, NUM_HEADS)
for layer in range(NUM_BLOCKS):
    ckpt = run_until_heads(input_ids, layer)
    other = run_until_heads(other_ids, layer)
    for h in range(NUM_HEADS):
        _, f, g = chances(resume_with_head(ckpt, h, other["head_out"][0, h, LAST_POS]))
        french_after_swap[layer, h], german_after_swap[layer, h] = f, g

print("Chance of ' French' (%) after swapping one head of ' speak' with its Berlin version")
print(f"{'Block':<7}" + "".join(f"{'h' + str(h):>5}" for h in range(NUM_HEADS)))
for layer in range(NUM_BLOCKS):
    print(f"{layer:<7}" + "".join(f"{100 * f:>5.0f}" for f in french_after_swap[layer]))

Chance of ' French' (%) after swapping one head of ' speak' with its Berlin version
Block     h0   h1   h2   h3   h4   h5   h6   h7   h8   h9  h10  h11
0         63   63   63   63   63   63   64   63   63   63   63   63
1         63   63   63   63   63   63   63   64   63   63   63   63
2         63   65   63   63   63   63   63   63   63   63   63   63
3         63   63   63   63   63   63   63   63   63   63   63   63
4         63   63   63   63   63   63   63   63   64   63   63   63
5         63   63   63   63   63   63   63   63   63   63   63   63
6         63   63   63   63   63   63   63   63   63   63   63   63
7         63   63   63   63   63   63   63   63   63   63   63   63
8         63   63   64   63   63   63   63   63   64   63   62   53
9         63   63   63   63   63   62   63   63   63   63   63   63
10        59   63   63   65   63   63   64   67   63   64   60   63
11        63   63   26   61   63   64   63   64   62   63   70   63


## Step 12: The heads that matter most
We rank all 144 heads by how much swapping them alone lowers the chance of ' French'.

In [12]:
drop = NORMAL_FRENCH - french_after_swap
ranked = [(divmod(idx.item(), NUM_HEADS), drop.flatten()[idx].item()) for idx in drop.flatten().argsort(descending=True)]

print(f"{'Rank':<6}{'Block.head':<12}{'French drop':>12}{'French':>9}{'German':>9}")
for rank, ((layer, h), d) in enumerate(ranked[:10], 1):
    print(f"{rank:<6}{f'{layer}.{h}':<12}{d:>12.1%}{french_after_swap[layer, h]:>9.1%}{german_after_swap[layer, h]:>9.1%}")
print(f"\nHeads that lower ' French' by more than 5 points: {(drop > 0.05).sum().item()} of {drop.numel()}")
print(f"Heads that raise ' French' by more than 5 points: {(drop < -0.05).sum().item()} of {drop.numel()}")

Rank  Block.head   French drop   French   German
1     11.2               37.0%    26.0%    10.2%
2     8.11                9.7%    53.4%     3.0%
3     10.0                4.2%    58.8%     1.6%
4     10.10               3.0%    60.1%     0.9%
5     11.3                2.2%    60.8%     0.9%
6     8.10                1.1%    61.9%     0.6%
7     11.8                0.7%    62.3%     0.7%
8     9.5                 0.6%    62.5%     0.6%
9     11.11               0.5%    62.5%     0.7%
10    11.6                0.5%    62.5%     0.6%

Heads that lower ' French' by more than 5 points: 2 of 144
Heads that raise ' French' by more than 5 points: 1 of 144


## Step 13: Change the top heads together
We pause after the heads of **each** block that holds a chosen head, in the same run, and swap in the Berlin version of the chosen heads for ' speak'.

In [13]:
def swap_heads(chosen):
    """Edits for run_model: in each block, replace the chosen heads' ' speak' rows with their Berlin versions."""
    other_heads = {}
    state = embed(other_ids)
    for i in range(NUM_BLOCKS):
        _, head_out = attend(*qkv(i, state))
        other_heads[i] = head_out
        state = think_part(i, merge_part(i, state, head_out))

    edits = {}
    for layer in {layer for layer, _ in chosen}:
        heads = [h for l, h in chosen if l == layer]
        def edit(head_out, layer=layer, heads=heads):
            head_out = head_out.clone()
            head_out[0, heads, LAST_POS] = other_heads[layer][0, heads, LAST_POS]
            return head_out
        edits[layer] = edit
    return edits

print(f"{'Heads swapped':<15}{'top word':<12}{'French':>8}{'German':>8}")
for k in [1, 2, 3, 5, 10, 20]:
    edits = swap_heads([head for head, _ in ranked[:k]])
    parts_run.clear()
    logits = run_model(input_ids, edits=edits)
    assert len(parts_run) == 3 * NUM_BLOCKS, "Each part should run exactly once"
    top, f, g = chances(logits)
    print(f"{'top ' + str(k):<15}{top!r:<12}{f:>8.1%}{g:>8.1%}")

all_heads = [(l, h) for l in range(NUM_BLOCKS) for h in range(NUM_HEADS)]
top, f, g = chances(run_model(input_ids, edits=swap_heads(all_heads)))
print(f"{'all 144':<15}{top!r:<12}{f:>8.1%}{g:>8.1%}")

TOP5 = [head for head, _ in ranked[:5]]
print("\nTop 5 heads:", ", ".join(f"{l}.{h}" for l, h in TOP5))
compare(run_model(input_ids, edits=swap_heads(TOP5)), "Top 5 heads swapped")

Heads swapped  top word      French  German
top 1          ' French'      26.0%   10.2%
top 2          ' German'      10.6%   28.8%
top 3          ' German'       3.9%   44.6%
top 5          ' German'       2.1%   53.0%
top 10         ' German'       1.9%   50.5%
top 20         ' German'       0.3%   63.8%
all 144        ' German'       0.8%   63.6%

Top 5 heads: 11.2, 8.11, 10.0, 10.10, 11.3
Normal model              Top 5 heads swapped
  ' French'    63.05%      ' German'    53.01%
  ' English'    4.93%      ' English'    5.75%
  ' to'         3.16%      ' to'         5.24%
  ' a'          2.94%      ' a'          3.69%
  ' with'       1.76%      ' the'        2.33%

  ' French': 63.05% ->  2.14%
  ' German':  0.62% -> 53.01%


---
# Part C: block 11, changing where the heads look
In Experiment 4, replacing only the **keys** of ' Paris' in block 11 with the Berlin ones made ' French' go **up** from 63% to 83%.
A key only decides how much attention a word gets, not what it hands over. So our guess was:
*the Berlin key makes the heads of ' speak' look **more** at ' Paris', while what they collect there is still Paris information.*

## Step 14: PAUSE inside block 11, swap the keys of ' Paris', and see where the heads look

In [14]:
PAUSE_C = 11

ckpt_c = run_until_heads(input_ids, PAUSE_C)  # <-- PAUSE
q, k, v = qkv(PAUSE_C, ckpt_c["before_look"])
_, k_berlin, v_berlin = qkv(PAUSE_C, run_until_heads(other_ids, PAUSE_C)["before_look"])

k_swapped = k.clone()
k_swapped[0, :, PARIS_POS] = k_berlin[0, :, PARIS_POS]  # <-- MODIFY: only the keys of ' Paris'
weights_swapped, head_out_swapped = attend(q, k_swapped, v)

print(f"Block {PAUSE_C}, word ' speak': share of attention given to ' Paris'")
print(f"{'Head':<6}{'normal':>8}{'Berlin key':>12}{'French push (normal -> Berlin key)':>38}")
for h in range(NUM_HEADS):
    w0 = ckpt_c["weights"][0, h, LAST_POS, PARIS_POS].item()
    w1 = weights_swapped[0, h, LAST_POS, PARIS_POS].item()
    p0 = french_push(PAUSE_C, h, ckpt_c["head_out"][0, h, LAST_POS])
    p1 = french_push(PAUSE_C, h, head_out_swapped[0, h, LAST_POS])
    print(f"{h:<6}{w0:>8.0%}{w1:>12.0%}{p0:>22.2f} -> {p1:<8.2f}")
print(f"\nAverage over 12 heads: {ckpt_c['weights'][0, :, LAST_POS, PARIS_POS].mean():.1%} -> "
      f"{weights_swapped[0, :, LAST_POS, PARIS_POS].mean():.1%}")

Block 11, word ' speak': share of attention given to ' Paris'
Head    normal  Berlin key    French push (normal -> Berlin key)
0          12%         14%                  2.39 -> 2.35    
1           6%          8%                  0.38 -> 0.36    
2          28%         54%                 18.78 -> 33.98   
3          24%         21%                  2.67 -> 2.52    
4           4%          7%                  0.52 -> 0.50    
5           4%          4%                  0.44 -> 0.48    
6          13%         17%                  0.40 -> 0.60    
7           4%          5%                 -0.04 -> -0.11   
8           5%          6%                  1.97 -> 1.97    
9           5%          4%                 -0.41 -> -0.36   
10         38%         25%                 -6.55 -> -4.70   
11          4%          6%                 -0.31 -> -0.26   

Average over 12 heads: 12.3% -> 14.3%


## Step 15: RESUME with the changed heads
We merge the heads computed with the Berlin key and finish the run. This should reproduce Experiment 4's result (about 83% French).
Then, for comparison, we also swap the **values** of ' Paris' (what it hands over), and both.

In [15]:
v_swapped = v.clone()
v_swapped[0, :, PARIS_POS] = v_berlin[0, :, PARIS_POS]

print(f"{'Swapped in block 11':<22}{'top word':<12}{'French':>8}{'German':>8}")
for name, keys, vals in [("nothing", k, v), ("keys only", k_swapped, v), ("values only", k, v_swapped),
                         ("keys and values", k_swapped, v_swapped)]:
    _, head_out = attend(q, keys, vals)
    parts_run.clear()
    logits = resume_after_heads(copy_ckpt(ckpt_c, head_out=head_out))  # <-- RESUME
    assert parts_run == expected_after(PAUSE_C)
    top, f, g = chances(logits)
    print(f"{name:<22}{top!r:<12}{f:>8.1%}{g:>8.1%}")

Swapped in block 11   top word      French  German
nothing               ' French'      63.0%    0.6%
keys only             ' French'      83.1%    0.2%
values only           ' French'      41.4%    3.3%
keys and values       ' French'      33.9%   12.0%
